Imports, podem alterar conforme necessário

In [ ]:
import seaborn as sns
import pandas as pd
import numpy as np

#### Lendo base de dados e removendo valores aleatoriamente **executar SEM ALTERAR**

In [ ]:
#Carregando base de dados
penguins = sns.load_dataset('penguins')

np.random.seed(42)  #Definindo seed do random para replicabilidade

#Removendo valores
removidos = set()
porcentagem = 0.30 #Porcentagem (0~1) das células a serem removidas
qtdCelulas = len(penguins)*(len(penguins.columns)) #Quantidade de células na base de dados, ignorando a última coluna

for i in range(int(np.ceil(porcentagem*qtdCelulas))):
  linha = np.random.randint(0, len(penguins))
  coluna = np.random.randint(0, len(penguins.columns))
  while (linha, coluna) in removidos:
    linha = np.random.randint(0, len(penguins))
    coluna = np.random.randint(0, len(penguins.columns))

  penguins.iloc[linha, coluna] = np.nan
  removidos.add((linha,coluna))

penguins.info()
print("\n\nForam removidas ",str(len(removidos)), "células das ",str(qtdCelulas))
del removidos
del qtdCelulas
del porcentagem

# **Podem modificar daqui para baixo para fazer a imputação de valores**

# Trabalho 2 - Imputacao de Dados: Penguins

Aqui eu trato a base Penguins que teve 30% das celulas removidas aleatoriamente.
O objetivo e imputar os valores faltantes de forma justificada, sem distorcer os dados.

Colunas numericas: bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g
Colunas categoricas: species, island, sex

Importando o que eu preciso alem do que ja foi importado acima

In [ ]:
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler

# 1. Analise Exploratoria Inicial

Aqui eu olho a base antes de fazer qualquer coisa, pra entender o que eu tenho e o que falta

In [ ]:
penguins.info()

In [ ]:
penguins.describe()

In [ ]:
# isnull() marca True onde tem NaN, sum() conta esses True
# resultado: quantos valores faltam em cada coluna
print(penguins.isnull().sum())

Cerca de 30% de cada coluna esta faltando, que e exatamente o que foi removido.
Nenhuma coluna esta completamente vazia, entao da pra imputar.

Aqui eu ploto os histogramas das colunas numericas lado a lado.
plt.subplot(linhas, colunas, posicao) divide a figura em uma grade e posiciona cada grafico nela.

In [ ]:
# plt.figure define o tamanho da figura inteira
# plt.subplot(2, 2, 1) = grade 2x2, posicao 1 (canto superior esquerdo)
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
sns.histplot(data=penguins, x='bill_length_mm')
plt.title('bill_length_mm')

plt.subplot(2, 2, 2)
sns.histplot(data=penguins, x='bill_depth_mm')
plt.title('bill_depth_mm')

plt.subplot(2, 2, 3)
sns.histplot(data=penguins, x='flipper_length_mm')
plt.title('flipper_length_mm')

plt.subplot(2, 2, 4)
sns.histplot(data=penguins, x='body_mass_g')
plt.title('body_mass_g')

plt.suptitle('Distribuicao das variaveis numericas (antes da imputacao)')
plt.tight_layout()
plt.show()

As distribuicoes tem mais de um pico (sao multimodais), o que indica que cada especie forma
um grupo diferente. Por isso a media global nao seria uma boa escolha pra imputar,
ela cairia num valor que nao representa nenhum pinguim real.

Aqui eu ploto a frequencia das colunas categoricas lado a lado

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 3, 1)
sns.histplot(data=penguins, x='species')
plt.title('species')

plt.subplot(1, 3, 2)
sns.histplot(data=penguins, x='island')
plt.title('island')

plt.subplot(1, 3, 3)
sns.histplot(data=penguins, x='sex')
plt.title('sex')

plt.suptitle('Distribuicao das variaveis categoricas (antes da imputacao)')
plt.tight_layout()
plt.show()

Essas proporcoes sao minha referencia. Depois da imputacao eu comparo
pra ver se nao distorci nada.

Aqui eu ploto boxplots por especie pra ver se cada uma ocupa uma faixa de valores diferente

In [ ]:
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
sns.boxplot(data=penguins, x='species', y='bill_length_mm')
plt.title('bill_length_mm')

plt.subplot(2, 2, 2)
sns.boxplot(data=penguins, x='species', y='bill_depth_mm')
plt.title('bill_depth_mm')

plt.subplot(2, 2, 3)
sns.boxplot(data=penguins, x='species', y='flipper_length_mm')
plt.title('flipper_length_mm')

plt.subplot(2, 2, 4)
sns.boxplot(data=penguins, x='species', y='body_mass_g')
plt.title('body_mass_g')

plt.suptitle('Boxplots por especie')
plt.tight_layout()
plt.show()

Cada especie ocupa uma faixa bem diferente em todas as medidas.
Isso vai justificar o uso do KNN: ele encontra vizinhos que provavelmente
sao da mesma especie, dando uma estimativa mais precisa do que a media global.

Aqui eu vejo a correlacao entre as colunas numericas

In [ ]:
# .corr() calcula o quanto duas colunas se relacionam, de -1 a 1
# proximo de 1: quando uma sobe, a outra sobe tambem
# isso e o que o KNN precisa pra funcionar bem
colunas_numericas = ['bill_length_mm', 'bill_depth_mm',
                     'flipper_length_mm', 'body_mass_g']

print(penguins[colunas_numericas].corr())

As colunas sao correlacionadas entre si, especialmente flipper e body_mass.
Isso e bom pro KNN: ele usa as colunas conhecidas pra estimar a faltante.
Quanto mais correlacionadas, mais precisa a estimativa.

Aqui eu salvo uma copia da base com os NaN pra comparar depois e exporto pro Power BI (Dashboard 1)

In [ ]:
# .copy() cria uma copia independente
# sem copy, penguins_antes seria so um apelido de penguins
# qualquer mudanca em penguins mudaria penguins_antes tambem
penguins_antes = penguins.copy()

# decimal=',' e sep=';' exportam no formato brasileiro
# necessario pra Power BI ler os valores corretamente
penguins_antes.to_csv('penguins_inicial.csv', index=False, decimal=',', sep=';')
print("penguins_inicial.csv salvo")

# 2. Imputacao das Colunas Numericas

Antes de escolher o metodo, considerei 3 opcoes:

1. Remover as linhas com NaN
2. Imputar pela media ou mediana
3. Imputar pelo KNN

Remover as linhas seria a pior escolha: com 30% de valores faltando,
eu perderia mais de 100 amostras de uma base que ja tem so 344.
Isso reduziria muito a quantidade de dados disponíveis para treinar
algoritmos de machine learning, que e o objetivo final do tratamento.

Imputar pela media tambem nao funciona nesse caso. Os histogramas
acima mostram que as distribuicoes tem mais de um pico (sao multimodais).
Isso acontece porque cada especie ocupa uma faixa de valores diferente,
como confirmado pelos boxplots. A media global cairia entre esses grupos,
num valor que nao representa nenhum pinguim real. Por exemplo, a media
do comprimento do bico de todas as especies juntas ficaria num valor
que nenhuma especie de fato tem como valor tipico.

Por isso eu escolhi o KNN Imputer. A tabela de correlacao acima mostra
que as variaveis numericas sao correlacionadas entre si, especialmente
flipper_length_mm e body_mass_g. Isso significa que conhecendo algumas
medidas de um pinguim, eu consigo estimar as que estao faltando com
boa precisao. O KNN faz exatamente isso: encontra os 5 pinguins mais
parecidos nas colunas que existem e usa a media deles pra estimar
a coluna faltante. Como os vizinhos mais proximos provavelmente
sao da mesma especie, o valor estimado cai na faixa certa.

Sobre fit, transform e inverse_transform:
- fit: o algoritmo le os dados e memoriza o minimo e maximo de cada coluna
- transform: aplica a escala 0-1 usando o que foi memorizado
- fit_transform: faz os dois de uma vez
- inverse_transform: desfaz a escala, voltando pra gramas e mm

In [ ]:
# Passo 1: separar so as colunas numericas
dados_numericos = penguins[colunas_numericas]

# Passo 2: normalizar tudo entre 0 e 1
scaler = MinMaxScaler()
dados_normalizados = scaler.fit_transform(dados_numericos)

# Passo 3: preencher os NaN usando KNN com 5 vizinhos
imputer = KNNImputer(n_neighbors=5)
dados_imputados = imputer.fit_transform(dados_normalizados)

# Passo 4: voltar pra escala original (gramas, mm)
dados_imputados = scaler.inverse_transform(dados_imputados)

# Passo 5: colocar os valores imputados de volta no dataframe
penguins[colunas_numericas] = dados_imputados

print("Faltantes apos imputacao numerica:")
print(penguins[colunas_numericas].isnull().sum())

Usei n_neighbors=5 porque:
- Muito pequeno (ex: 1): muito sensivel a outliers
- Muito grande (ex: 50): vira praticamente uma media global, perdendo a vantagem do KNN
5 vizinhos e um valor equilibrado pra esse tamanho de base.

Aqui eu comparo os histogramas antes e depois da imputacao pra ver se a distribuicao foi preservada.
O alpha deixa os dois histogramas semitransparentes pra dar pra ver os dois ao mesmo tempo.
O label nomeia cada serie e o legend() mostra a caixinha de legenda no grafico.

In [ ]:
plt.figure(figsize=(12, 8))

plt.subplot(2, 2, 1)
sns.histplot(data=penguins_antes, x='bill_length_mm', color='red', label='Antes', alpha=0.5)
sns.histplot(data=penguins, x='bill_length_mm', color='blue', label='Depois', alpha=0.5)
plt.title('bill_length_mm')
plt.legend()

plt.subplot(2, 2, 2)
sns.histplot(data=penguins_antes, x='bill_depth_mm', color='red', label='Antes', alpha=0.5)
sns.histplot(data=penguins, x='bill_depth_mm', color='blue', label='Depois', alpha=0.5)
plt.title('bill_depth_mm')
plt.legend()

plt.subplot(2, 2, 3)
sns.histplot(data=penguins_antes, x='flipper_length_mm', color='red', label='Antes', alpha=0.5)
sns.histplot(data=penguins, x='flipper_length_mm', color='blue', label='Depois', alpha=0.5)
plt.title('flipper_length_mm')
plt.legend()

plt.subplot(2, 2, 4)
sns.histplot(data=penguins_antes, x='body_mass_g', color='red', label='Antes', alpha=0.5)
sns.histplot(data=penguins, x='body_mass_g', color='blue', label='Depois', alpha=0.5)
plt.title('body_mass_g')
plt.legend()

plt.suptitle('Antes vs Depois da imputacao numerica')
plt.tight_layout()
plt.show()

Os histogramas confirmam que a forma das distribuicoes foi preservada.
Os picos continuam nos mesmos lugares, a escala dos valores nao mudou
e nenhum valor absurdo foi inserido.

A tabela de estatisticas e a tabela de correlacao tambem confirmam isso:
as medias, desvios padrao e correlacoes entre colunas ficaram muito
proximos dos valores originais. Isso mostra que o KNN respeitou a
estrutura dos dados, algo que a media global nao conseguiria fazer.

# 3. Imputacao das Colunas Categoricas

Colunas categoricas nao tem media nem distancia numerica, entao o KNN nao se aplica direto.

A estrategia aqui e a moda condicional: em vez de pegar a categoria mais frequente do dataset inteiro,
eu pego a mais frequente dentro de um grupo relevante (por especie).

A ordem importa: eu imputo species primeiro, porque island e sex dependem dela.

Aqui eu imputo species pela moda global.

Para species eu usei a moda global porque e a primeira coluna a ser
imputada e eu nao tenho outro grupo para condicionar. Adelie e a especie
mais frequente na base, entao imputar com ela e conservador e
minimiza distorcao nas proporcoes do dataset.

Nao faria sentido usar as variaveis numericas para tentar inferir
a especie de forma mais sofisticada, pois essas mesmas variaveis
acabaram de ser imputadas e ja carregam alguma incerteza.
A moda global e a escolha mais segura nesse contexto.

In [ ]:
# mode() retorna o valor mais frequente, [0] pega o primeiro
moda_species = penguins['species'].mode()[0]
print('Especie mais frequente:', moda_species)

penguins['species'] = penguins['species'].fillna(moda_species)
print("Faltantes em species:", penguins['species'].isnull().sum())

Aqui eu vejo qual ilha e mais frequente em cada especie antes de imputar island.

Para island eu nao posso usar a moda global. A impressao abaixo mostra
que cada especie tem uma ilha dominante muito clara:
Chinstrap aparece quase so em Dream, Gentoo quase so em Biscoe.
Usar a moda global daria a mesma ilha pra todos os casos,
o que colocaria Gentoos em ilhas onde eles praticamente nao existem.

Por isso eu uso a moda por especie: pra cada pinguim com island faltando,
eu pego a ilha mais frequente da especie dele. Isso respeita
a relacao geografica real entre especie e ilha.

In [ ]:
print('Adelie    ->', penguins[penguins['species'] == 'Adelie']['island'].mode()[0])
print('Chinstrap ->', penguins[penguins['species'] == 'Chinstrap']['island'].mode()[0])
print('Gentoo    ->', penguins[penguins['species'] == 'Gentoo']['island'].mode()[0])

Aqui eu crio uma funcao pra imputar colunas categoricas pela moda da especie e aplico em island

In [ ]:
def imputar_por_especie(coluna):
    for especie in ['Adelie', 'Chinstrap', 'Gentoo']:
        mask_especie = penguins['species'] == especie
        moda = penguins.loc[mask_especie, coluna].mode()[0]
        mask_faltante = mask_especie & penguins[coluna].isna()
        penguins.loc[mask_faltante, coluna] = moda

imputar_por_especie('island')
print("Faltantes em island:", penguins['island'].isnull().sum())

Aqui eu vejo a relacao entre especie e sexo antes de imputar sex.

Para sex o raciocinio e parecido com island. O boxplot abaixo mostra
que machos e femeas tem distribuicoes de medidas corporais
levemente diferentes dentro de cada especie. A proporcao entre
machos e femeas pode variar de especie pra especie, entao
usar a moda global seria menos preciso do que a moda por especie.

Alem disso, manter consistencia no metodo de imputacao ao longo
de todas as colunas categoricas facilita a justificativa e
torna o processo mais organizado.

In [ ]:
# hue separa as caixas por sexo dentro de cada especie
sns.boxplot(data=penguins, x='species', y='bill_length_mm', hue='sex')
plt.title('Bico por especie e sexo')
plt.show()

In [ ]:
imputar_por_especie('sex')
print("Faltantes em sex:", penguins['sex'].isnull().sum())

In [ ]:
print("Valores faltantes apos toda a imputacao:")
print(penguins.isnull().sum())

# 4. Avaliacao Final

Aqui eu comparo as estatisticas antes e depois pra confirmar que a imputacao nao distorceu os dados

In [ ]:
print("Estatisticas ANTES:")
print(penguins_antes[colunas_numericas].describe())
print("\nEstatisticas DEPOIS:")
print(penguins[colunas_numericas].describe())

Aqui eu comparo a correlacao antes e depois pra ver se as relacoes entre colunas foram preservadas

In [ ]:
print("Correlacao ANTES:")
print(penguins_antes[colunas_numericas].corr())
print("\nCorrelacao DEPOIS:")
print(penguins[colunas_numericas].corr())

Aqui eu vejo as proporcoes categoricas depois da imputacao pra comparar com o que tinha antes

In [ ]:
plt.figure(figsize=(14, 5))

plt.subplot(1, 3, 1)
sns.histplot(data=penguins, x='species')
plt.title('species (depois)')

plt.subplot(1, 3, 2)
sns.histplot(data=penguins, x='island')
plt.title('island (depois)')

plt.subplot(1, 3, 3)
sns.histplot(data=penguins, x='sex')
plt.title('sex (depois)')

plt.suptitle('Distribuicao das categoricas apos imputacao')
plt.tight_layout()
plt.show()

As proporcoes se mantiveram coerentes com a base inicial.
As medias, desvios e correlacoes tambem ficaram proximos dos valores originais.
Isso confirma que a imputacao foi adequada.

Aqui eu exporto a base imputada pra usar no Power BI (Dashboard 2)

In [ ]:
penguins.to_csv('penguins_imputado.csv', index=False, decimal=',', sep=';')
print("penguins_imputado.csv salvo")
print("\nFaltantes restantes:")
print(penguins.isnull().sum())